# RCTB on the Huma-Num / CC-IN2P3 GPU platform

Runs the Rotary pipeline with a local vision model served by Ollama on the GPU node of this Jupyter session.

**Before the first run**, in a terminal of this session: `module load ollama`, check `ollama list`, and pull a vision model if cell 2 says none is available (`ollama pull qwen3-vl:32b`).

**How a run works.** Cell 3 starts the pipeline as a background process that keeps going when the browser tab closes. Every finished page or document is written to disk at once. When the session ends (23 h limit) the process is killed with it; open a new session and run cells 1 to 3 again: finished work is skipped and the run continues where it stopped. Nothing needs to be split into batches by hand.

Run the cells in order. Cells 4 to 6 can be used at any time.

In [ ]:
# 1. Settings
import json, os, re, subprocess, sys, time
from pathlib import Path

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()   # folder containing config.hpc.yaml
CONFIG = "config.hpc.yaml"
FILES = ["B37-F01"]     # pilot: one folder. Set to None for the whole collection.
PAGES = None            # e.g. [1, 4, 9] to transcribe only these pages of the pilot folder (quick trial); None = all pages
STAGE = "all"           # all = inventory, pages, transcribe, segment, enrich, normalize, build, embed (each skips finished work)

assert (PROJECT_DIR / CONFIG).exists(), f"{CONFIG} not found in {PROJECT_DIR}: set PROJECT_DIR"
LOG_PATH = PROJECT_DIR / "data" / "work" / "run.log"
PID_PATH = PROJECT_DIR / "data" / "work" / "run.pid"
LOG_PATH.parent.mkdir(parents=True, exist_ok=True)

def resolve_ollama_host() -> str:
    """OLLAMA_HOST of this kernel, else the one `module load ollama` sets in a login shell (the port differs per user)."""
    val = os.environ.get("OLLAMA_HOST", "")
    if not val:
        try:
            raw = subprocess.run(["bash", "-lc", "module load ollama >/dev/null 2>&1; echo $OLLAMA_HOST"],
                                 capture_output=True, text=True, timeout=30).stdout
            lines = [l.strip() for l in raw.splitlines() if l.strip()]
            if lines and re.match(r"^(https?://)?[\w.\-]+:\d+$", lines[-1]):
                val = lines[-1]
        except Exception:
            pass
    val = val or "127.0.0.1:11434"
    return val if val.startswith("http") else f"http://{val}"

OLLAMA_HOST = resolve_ollama_host()
ENV = {**os.environ, "OLLAMA_HOST": OLLAMA_HOST, "PYTHONUNBUFFERED": "1"}

# Python packages missing from the platform's kernel live in a folder inside the project (installed by cell 2b)
DEPS = PROJECT_DIR / ".pydeps"
if DEPS.exists():
    sys.path.insert(0, str(DEPS))
    ENV["PYTHONPATH"] = str(DEPS) + os.pathsep + ENV.get("PYTHONPATH", "")

def rctb(*args, check=False):
    """Run `python -m rctb ...` in the project folder and show its output."""
    r = subprocess.run([sys.executable, "-m", "rctb", *args, "--config", CONFIG], cwd=PROJECT_DIR, env=ENV,
                       capture_output=True, text=True)
    print(r.stdout + r.stderr)
    if check and r.returncode:
        raise RuntimeError(f"rctb {args[0]} failed")
    return r

print(f"Project: {PROJECT_DIR}\nOllama:  {OLLAMA_HOST}\nFolders: {FILES or 'all'}")

In [ ]:
# 2. Checks: Python packages, Ollama, a vision model, the input files
import importlib.util, shutil, urllib.request, yaml

missing = [m for m in ("yaml", "pydantic", "pandas", "pyarrow", "sklearn", "openpyxl", "pypdf", "PIL") if importlib.util.find_spec(m) is None]
print("Python packages:", "ok" if not missing else f"MISSING {missing} -> run cell 2b once, then this cell again")

def ollama(path, body=None):
    req = urllib.request.Request(OLLAMA_HOST + path, data=json.dumps(body).encode() if body else None,
                                 headers={"Content-Type": "application/json"})
    with urllib.request.build_opener(urllib.request.ProxyHandler({})).open(req, timeout=20) as r:
        return json.loads(r.read())

cfg = yaml.safe_load((PROJECT_DIR / CONFIG).read_text())
wanted = {cfg["llm"]["model"]} | {v["model"] for v in (cfg["llm"].get("stages") or {}).values() if v.get("model")}
try:
    names = [m["name"] for m in ollama("/api/tags")["models"]]
    print(f"Ollama reachable. Pulled models and what they accept:")
    vision = {}
    for n in names:
        caps = ollama("/api/show", {"model": n}).get("capabilities", [])
        vision[n] = "vision" in caps
        print(f"   {n:<28} {'VISION' if vision[n] else 'text only':<10} {caps}")
    for w in sorted(wanted):
        if w not in names:
            print(f"!! {w} (config) is not pulled -> terminal: ollama pull {w}")
        elif not vision[w]:
            print(f"!! {w} (config) does not accept images: use it only for a stage with `images: false`")
        else:
            print(f"OK {w} will be used")
    emb = cfg.get("embedding", {})
    if emb.get("backend") == "ollama":
        em = emb.get("ollama_model", "bge-m3")
        print(f"OK embedding model {em}" if any(n.split(":")[0] == em.split(":")[0] for n in names)
              else f"!! embedding model {em} is not pulled -> terminal: ollama pull {em}   (until then the search index uses the tfidf baseline)")
except Exception as e:
    print(f"!! Ollama not reachable at {OLLAMA_HOST} ({e}). In a terminal on this node: module load ollama; ollama list")

if missing:
    raise SystemExit("Install the missing packages with cell 2b, then run this cell again.")
for tool in ("pdftoppm", "tesseract"):
    print(f"{tool}: {'found' if shutil.which(tool) else 'not on this machine (fine if your PNG + text folders are in data/raw/pages)'}")
if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.used,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout)
rctb("inventory");

In [ ]:
# 2b. Only if cell 2 reports MISSING packages: install them into the project folder (once; kept between sessions)
PIP_NAMES = {"yaml": "pyyaml", "pydantic": "pydantic>=2", "pandas": "pandas", "pyarrow": "pyarrow", "sklearn": "scikit-learn",
             "openpyxl": "openpyxl", "pypdf": "pypdf", "PIL": "pillow"}
todo = [PIP_NAMES[m] for m in PIP_NAMES if importlib.util.find_spec(m) is None]
if not todo:
    print("Nothing to install.")
else:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--target", str(DEPS), *todo], capture_output=True, text=True)
    print(r.stdout[-1500:], r.stderr[-1500:])
    if r.returncode == 0:
        sys.path.insert(0, str(DEPS))
        ENV["PYTHONPATH"] = str(DEPS) + os.pathsep + ENV.get("PYTHONPATH", "")
        importlib.invalidate_caches()
        print(f"Installed {todo} into {DEPS}. Now run cell 2 again.")
    else:
        print("Installation failed. If the message mentions the network, this node has no internet access: "
              "run the same command in a terminal on a node that has it:\n"
              f"  {sys.executable} -m pip install --target {DEPS} " + " ".join(f"'{t}'" for t in todo))

In [ ]:
# 3. Start (or continue) the run in the background
def running_pid():
    """PID of the background run if it is alive (a finished process that was not yet collected does not count)."""
    try:
        pid = int(PID_PATH.read_text())
        try:
            os.waitpid(pid, os.WNOHANG)          # collect it if it was started by this kernel and has ended
        except ChildProcessError:
            pass
        os.kill(pid, 0)
        state = Path(f"/proc/{pid}/stat").read_text().rsplit(")", 1)[1].split()[0]
        return None if state == "Z" else pid
    except Exception:
        return None

if running_pid():
    print(f"Already running (PID {running_pid()}). Use cell 4 to watch it, cell 6 to stop it.")
else:
    cmd = [sys.executable, "-m", "rctb", STAGE, "--config", CONFIG] + (["--files", *FILES] if FILES else [])
    if PAGES:
        cmd += ["--pages", *map(str, PAGES)]
    log = open(LOG_PATH, "a", buffering=1)
    log.write(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} start: {' '.join(cmd[1:])} | host {os.uname().nodename} | {OLLAMA_HOST} =====\n")
    proc = subprocess.Popen(cmd, cwd=PROJECT_DIR, env=ENV, stdout=log, stderr=subprocess.STDOUT,
                            stdin=subprocess.DEVNULL, start_new_session=True)
    PID_PATH.write_text(str(proc.pid))
    print(f"Started PID {proc.pid}; log: {LOG_PATH}\nIt keeps running if this tab is closed, and stops when the session ends.")

In [ ]:
# 4. Watch the log (interrupting this cell does NOT stop the run)
with open(LOG_PATH, encoding="utf-8") as f:
    f.seek(max(0, LOG_PATH.stat().st_size - 3000))
    try:
        while True:
            line = f.readline()
            if line:
                print(line, end="")
            elif not running_pid():
                print("\n[run finished or stopped]")
                break
            else:
                time.sleep(3)
    except KeyboardInterrupt:
        print("\n[stopped watching; the run continues]")

In [ ]:
# 5. Progress: what is done, what is left, speed, failures
print("Run is", f"active (PID {running_pid()})" if running_pid() else "not active")
rctb("status", *(["--files", *FILES] if FILES else []));

In [ ]:
# 6. Stop the run (safe at any time: finished work is kept, restart with cell 3)
import signal
pid = running_pid()
if pid:
    os.killpg(os.getpgid(pid), signal.SIGTERM)
    time.sleep(2)
    print(f"Stopped PID {pid}" if not running_pid() else f"PID {pid} is still shutting down; run this cell again")
else:
    print("Nothing is running.")

In [ ]:
# 7. After the pilot: score the segmentation against the reference and look at the records
rctb("evaluate", "gold/B37-F01.yaml")   # also writes data/out/eval/B37-F01_segmentation.html: open it to see every disagreement
sys.path.insert(0, str(PROJECT_DIR))
import pandas as pd
from rctb.search import TextBase
tb = TextBase(PROJECT_DIR / cfg["paths"]["out_dir"])
pd.set_option("display.max_colwidth", 90)
tb.sql("SELECT doc_id, genre, date_produced, title, substr(summary, 1, 200) AS summary FROM documents WHERE file_number = 'B37-F01'")

In [ ]:
# 8. Optional: compare models on the pilot folder (foreground; each model gets its own output folders)
MODELS = ["qwen3.5:35b", "qwen3-vl:32b"]     # must be pulled and accept images (see cell 2)
PILOT = "B37-F01"

results = {}
for model in MODELS:
    tag = re.sub(r"[^\w]+", "_", model)
    c = yaml.safe_load((PROJECT_DIR / CONFIG).read_text())
    c["llm"]["model"] = model
    c["paths"]["work_dir"], c["paths"]["out_dir"] = f"data/compare/{tag}/work", f"data/compare/{tag}/out"
    cfg_path = PROJECT_DIR / f"config.compare.{tag}.yaml"
    cfg_path.write_text(yaml.safe_dump(c, allow_unicode=True))
    for stage in ("inventory", "pages", "transcribe", "segment", "enrich", "build"):
        r = subprocess.run([sys.executable, "-m", "rctb", stage, "--config", cfg_path.name, "--files", PILOT],
                           cwd=PROJECT_DIR, env=ENV, capture_output=True, text=True)
        print(f"[{model}] {(r.stdout + r.stderr).strip().splitlines()[-1] if (r.stdout + r.stderr).strip() else stage}")
    r = subprocess.run([sys.executable, "-m", "rctb", "evaluate", f"gold/{PILOT}.yaml", "--config", cfg_path.name],
                       cwd=PROJECT_DIR, env=ENV, capture_output=True, text=True)
    results[model] = r.stdout.strip().splitlines()[-1] if r.stdout.strip() else r.stderr[-300:]
for m, res in results.items():
    print(f"\n{m}\n  {res}")
print("\nRead the records side by side in data/compare/<model>/out/json/ before choosing: boundary scores are only one criterion.")